# Análise de dados TCP-CI

In [26]:
import pandas as pd

In [27]:
df = pd.read_csv('./T CELL/DENV 1 - T Cell Prediction - Class I.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,VTGKTIHEW,303,311,9,HLA-B*57:01,303,0.01,VTGKTIHEW,VTGKTIHEW,0.997406,0.01
1,1,VTGKTIHEW,303,311,9,HLA-B*58:01,303,0.01,VTGKTIHEW,VTGKTIHEW,0.996622,0.01
2,1,SEMIIPKIY,239,247,9,HLA-B*44:03,239,0.01,SEMIIPKIY,SEMIIPKIY,0.996239,0.01
3,1,SEMIIPKIY,239,247,9,HLA-B*44:02,239,0.01,SEMIIPKIY,SEMIIPKIY,0.992169,0.01
4,1,LSAAIGKAW,42,50,9,HLA-B*57:01,42,0.01,LSAAIGKAW,LSAAIGKAW,0.989554,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
36985,1,PVKEKEENLVKS,337,348,12,HLA-B*53:01,1366,100.00,PVEENLVKS,PVKEKEENLVKS,0.000000,100.00
36986,1,EKEENLVKSMVS,340,351,12,HLA-A*11:01,1369,100.00,ENLVKSMVS,EKEENLVKSMVS,0.000000,100.00
36987,1,EKEENLVKSMVS,340,351,12,HLA-A*24:02,1369,100.00,EKEENKSMV,EKEENLVKSMV,0.000000,100.00
36988,1,EKEENLVKSMVS,340,351,12,HLA-A*32:01,1369,100.00,ENLVKSMVS,EKEENLVKSMVS,0.000000,100.00


## Selecionando Epítopos com median binding percentile menor que 5.

In [28]:
df_mbp_m5 = df[df['median binding percentile'] < 5].copy()
df_mbp_m5

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,VTGKTIHEW,303,311,9,HLA-B*57:01,303,0.01,VTGKTIHEW,VTGKTIHEW,0.997406,0.01
1,1,VTGKTIHEW,303,311,9,HLA-B*58:01,303,0.01,VTGKTIHEW,VTGKTIHEW,0.996622,0.01
2,1,SEMIIPKIY,239,247,9,HLA-B*44:03,239,0.01,SEMIIPKIY,SEMIIPKIY,0.996239,0.01
3,1,SEMIIPKIY,239,247,9,HLA-B*44:02,239,0.01,SEMIIPKIY,SEMIIPKIY,0.992169,0.01
4,1,LSAAIGKAW,42,50,9,HLA-B*57:01,42,0.01,LSAAIGKAW,LSAAIGKAW,0.989554,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
2496,1,LSAAIGKAW,42,50,9,HLA-A*24:02,42,4.90,LSAAIGKAW,LSAAIGKAW,0.003380,4.90
2497,1,TCIWPKSHTLWS,222,233,12,HLA-A*24:02,1251,4.90,TWPKSHTLW,TCIWPKSHTLW,0.003356,4.90
2498,1,SWKSWGKAKII,114,124,11,HLA-A*24:02,801,4.90,SWWGKAKII,SWKSWGKAKII,0.003267,4.90
2499,1,SEKNETWKLARA,204,215,12,HLA-B*44:03,1233,4.90,SEKNETWKA,SEKNETWKLARA,0.003159,4.90


## Agrupando por pepitideos e agregando colunas pertinentes.

In [29]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    ))

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIGKAWEEGV,44,54,3,2.90,"HLA-A*02:01, HLA-A*02:06, HLA-A*68:02"
1,AAIKDSKAV,186,194,7,2.60,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*6..."
2,AAIKDSKAVH,186,195,2,3.50,"HLA-A*30:02, HLA-B*15:01"
3,ADMGYWIESEK,196,206,2,4.15,"HLA-A*03:01, HLA-A*11:01"
4,ADSPKRLSA,36,44,3,3.20,"HLA-B*08:01, HLA-B*40:01, HLA-B*44:02"
...,...,...,...,...,...,...
597,YTQVCDHRL,175,183,8,2.95,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:06, HLA-A*2..."
598,YTQVCDHRLM,175,184,4,3.45,"HLA-A*01:01, HLA-B*15:01, HLA-B*57:01, HLA-B*5..."
599,YTQVCDHRLMSA,175,186,1,2.50,HLA-A*01:01
600,YWIESEKNETW,200,210,10,0.80,"HLA-A*23:01, HLA-A*24:02, HLA-A*32:01, HLA-B*3..."


# Filtragem por epítopos presentes em mais de determinada quantidade de alelos.

In [23]:
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= 10
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,ATRLENIMW,60,68,13,3.300,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
1,AVHADMGYW,193,201,10,1.645,"HLA-A*23:01, HLA-A*26:01, HLA-A*30:02, HLA-A*3..."
2,CIWPKSHTL,223,231,20,1.600,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."
3,CTLPPLRFK,316,324,12,1.850,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
4,ECPDNQRAW,142,150,11,1.900,"HLA-A*23:01, HLA-A*24:02, HLA-A*26:01, HLA-A*3..."
5,ESEMIIPKIY,238,247,10,2.550,"HLA-A*01:01, HLA-A*26:01, HLA-A*30:02, HLA-B*3..."
6,ETWKLARASF,208,217,13,2.800,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
7,EVHTWTEQY,24,32,16,1.500,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
8,EVHTWTEQYKF,24,34,10,3.250,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
9,FQADSPKRL,34,42,18,2.450,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."


In [25]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["qte_de_alelos", "median_binding_percentile"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,HTWTEQYKF,26,34,21,1.500,"HLA-A*01:01, HLA-A*02:06, HLA-A*11:01, HLA-A*2..."
1,CIWPKSHTL,223,231,20,1.600,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."
2,RPQPMEHKY,105,113,19,1.400,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
3,ISNELNHIL,71,79,19,2.600,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
4,FQADSPKRL,34,42,18,2.450,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."
5,FVTNEVHTW,20,28,17,1.200,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:06, HLA-A*2..."
6,ILLENDMKF,78,86,17,1.200,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
7,QPMEHKYSW,107,115,16,0.545,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
8,EVHTWTEQY,24,32,16,1.500,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
9,FTTNIWLKL,163,171,16,1.650,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
